# Latih Ulang Model SVM untuk Klasifikasi Risiko Gempa Bumi Sesar Lembang (Data Riil USGS)

Notebook ini digunakan untuk melatih ulang model *Support Vector Machine* (SVM) menggunakan data riil katalog kegempaan USGS Jawa Barat tahun 1990–2026. Proses pemodelan ini mencakup:
1. Pengunggahan seluruh berkas CSV periodik USGS sekaligus dari laptop ke Colab
2. Penggabungan data, pembersihan, & penapisan spasial regional Jawa Barat
3. Pelabelan risiko gempa bumi dangkal merusak secara objektif
4. Normalisasi fitur dan pembagian dataset
5. Optimasi hyperparameter (*Grid Search*)
6. Evaluasi performa secara aman tanpa error kelas kosong
7. Ekspor model biner `.pkl` untuk integrasi ke backend FastAPI
8. Unduh otomatis seluruh file gambar visualisasi dan model biner ke laptop Anda

In [ ]:
# Import seluruh pustaka yang diperlukan
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from io import StringIO
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, precision_score, recall_score, f1_score
import folium
from folium.plugins import HeatMap

# Atur visualisasi grafik
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 11

print('✅ Seluruh pustaka berhasil dimuat!')

In [ ]:
# Langkah Pengunggahan Seluruh Berkas CSV USGS dari Laptop Anda
from google.colab import files

print('📁 Silakan unggah semua berkas CSV periodik USGS Anda (pilih semua berkas sekaligus: 1990-1994.csv, 1995-1999.csv, dst.)...')
uploaded = files.upload()

dfs_uploaded = []
for fname, content in uploaded.items():
    # Coba muat dengan berbagai separator pembatas
    for sep in [',', ';', '\t']:
        try:
            df_temp = pd.read_csv(StringIO(content.decode('utf-8')), sep=sep)
            if len(df_temp.columns) >= 4:
                print(f'  ✅ Berhasil memuat {fname} | {len(df_temp):,} baris | sep="{sep}"')
                dfs_uploaded.append(df_temp)
                break
        except:
            continue

if dfs_uploaded:
    df_raw = pd.concat(dfs_uploaded, ignore_index=True)
    # Bersihkan duplikat berdasarkan waktu dan koordinat jika ada
    df_raw = df_raw.drop_duplicates(subset=['time', 'latitude', 'longitude'] if 'time' in df_raw.columns else None)
    df_raw['source'] = 'USGS-Katalog-Manual-Upload'
    print(f'\n✅ Penggabungan selesai! Total data mentah: {len(df_raw):,} baris')
else:
    # Fallback ke folder lokal jika tidak berjalan di Colab
    print('⚠️ Tidak ada berkas yang diunggah. Mencoba memuat file lokal dari folder...')
    local_paths = [
        'data_lembang_dengan_prediksi (1).csv',
        'KUMPULAN DATA USGS UNTUK DIOLAH/data_lembang_dengan_prediksi (1).csv'
    ]
    for path in local_paths:
        if os.path.exists(path):
            df_raw = pd.read_csv(path)
            print(f'  ✅ Berhasil memuat lokal {path} | {len(df_raw):,} baris')
            break

if df_raw is not None:
    print(f'\n✅ Dataset USGS berhasil dimuat: {len(df_raw):,} baris')
else:
    raise ValueError('❌ Tidak ada data CSV USGS yang berhasil dimuat. Silakan unggah file Anda.')

## Pra-pemrosesan Data (Preprocessing)

In [ ]:
# Standardisasi nama kolom dan pembersihan data
df = df_raw.copy()
KOLOM_MAP = {
    'Tanggal': 'time', 'tgl': 'time', 'DateTime': 'time', 'datetime': 'time',
    'Lintang': 'latitude', 'lintang': 'latitude', 'Latitude': 'latitude',
    'Bujur': 'longitude', 'bujur': 'longitude', 'Longitude': 'longitude',
    'Kedalaman': 'depth', 'dalam': 'depth', 'Depth': 'depth',
    'Magnitude': 'mag', 'Magnitudo': 'mag', 'magnitude': 'mag'
}
df = df.rename(columns={k: v for k, v in KOLOM_MAP.items() if k in df.columns})

# Konversi tipe numerik secara aman
for col in ['latitude', 'longitude', 'depth', 'mag']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Bersihkan data kosong dan duplikat
df = df.dropna(subset=['mag', 'depth', 'latitude', 'longitude'])
df = df.drop_duplicates()

# Filter batas Indonesia secara makro
df = df[(df['mag'] >= 0) & (df['mag'] <= 10)]
df = df[(df['depth'] >= 0) & (df['depth'] <= 700)]

print(f'Data setelah pembersihan awal: {len(df):,} baris')

In [ ]:
# Filter spasial wilayah regional Jawa Barat sesuai batas proposal Bab III
LAT_MIN = -8.00
LAT_MAX = -5.50
LON_MIN = 106.00
LON_MAX = 109.00

df_lembang = df[
    (df['latitude'] >= LAT_MIN) & (df['latitude'] <= LAT_MAX) &
    (df['longitude'] >= LON_MIN) & (df['longitude'] <= LON_MAX)
].copy()

df_lembang = df_lembang.reset_index(drop=True)
print(f'✅ Filter spasial Jawa Barat aktif!')
print(f'   Total kejadian seismik riil USGS  : {len(df_lembang):,} event')
print(f'   Magnitudo terendah                : {df_lembang["mag"].min():.2f}')
print(f'   Magnitudo tertinggi               : {df_lembang["mag"].max():.2f}')
print(f'   Kedalaman hiposenter              : {df_lembang["depth"].min():.1f} - {df_lembang["depth"].max():.1f} km')

In [ ]:
# Visualisasi Distribusi Magnitudo, Kedalaman, dan Sebaran Spasial
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Magnitudo
axes[0].hist(df_lembang['mag'].dropna(), bins=30, color='seagreen', edgecolor='white')
axes[0].set_title('Distribusi Magnitudo (USGS)', fontweight='bold')
axes[0].set_xlabel('Magnitudo')
axes[0].set_ylabel('Frekuensi')
axes[0].axvline(x=3.0, color='orange', linestyle='--', label='M=3.0')
axes[0].axvline(x=5.0, color='red', linestyle='--', label='M=5.0')
axes[0].legend()

# Kedalaman
axes[1].hist(df_lembang['depth'].dropna(), bins=30, color='darkorange', edgecolor='white')
axes[1].set_title('Distribusi Kedalaman (USGS)', fontweight='bold')
axes[1].set_xlabel('Kedalaman (km)')
axes[1].axvline(x=60, color='red', linestyle='--', label='60 km')
axes[1].legend()

# Spasial
sc = axes[2].scatter(df_lembang['longitude'], df_lembang['latitude'],
                     c=df_lembang['mag'], cmap='YlOrRd', alpha=0.5, s=10)
plt.colorbar(sc, ax=axes[2], label='Magnitudo')
axes[2].set_title('Sebaran Spasial Gempa USGS', fontweight='bold')
axes[2].set_xlabel('Bujur (°BT)')
axes[2].set_ylabel('Lintang (°LS)')

plt.tight_layout()
plt.savefig('eda_distribusi_usgs.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Plot EDA USGS berhasil disimpan!')

In [ ]:
# Fungsi pelabelan risiko
def label_risiko(row):
    mag = row['mag']
    depth = row['depth']
    if mag > 5.0 and depth < 60:
        return 2
    elif 3.0 <= mag <= 5.0:
        return 1
    else:
        return 0

df_lembang['risk_label'] = df_lembang.apply(label_risiko, axis=1)
df_lembang['risk_name'] = df_lembang['risk_label'].map({0: 'Rendah', 1: 'Sedang', 2: 'Tinggi'})

# Plot sebaran kelas
dist = df_lembang['risk_name'].value_counts()
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'Rendah': '#2ecc71', 'Sedang': '#f39c12', 'Tinggi': '#e74c3c'}

dist.plot(kind='bar', ax=axes[0], color=[colors[c] for c in dist.index if c in colors], edgecolor='white', width=0.6)
axes[0].set_title('Jumlah Event per Kelas Risiko (USGS)', fontweight='bold')
axes[0].set_xlabel('Kelas Risiko')
axes[0].set_ylabel('Jumlah Kejadian')
axes[0].tick_params(rotation=0)
for bar_obj in axes[0].patches:
    axes[0].annotate(f'{int(bar_obj.get_height()):,}',
                     (bar_obj.get_x() + bar_obj.get_width() / 2., bar_obj.get_height()),
                     ha='center', va='bottom', fontsize=11, fontweight='bold')

dist.plot(kind='pie', ax=axes[1], colors=[colors[c] for c in dist.index if c in colors], autopct='%1.2f%%', startangle=90)
axes[1].set_title('Proporsi Kelas Risiko Gempa USGS', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.savefig('distribusi_kelas_usgs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ekstraksi Fitur (X) dan Label (y)
FITUR = ['mag', 'depth', 'latitude', 'longitude']
X = df_lembang[FITUR].values
y = df_lembang['risk_label'].values

# Inisiasi MinMaxScaler
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Pembagian dataset latih dan uji (80:20) secara stratified
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y if len(np.unique(y)) > 1 and np.min(np.bincount(y)) >= 2 else None
)

print('✅ Normalisasi & Pembagian Data Berhasil!')
print(f'   Jumlah Data Latih: {X_train.shape[0]} baris')
print(f'   Jumlah Data Uji  : {X_test.shape[0]} baris')

In [ ]:
# GridSearchCV untuk mencari parameter C dan Gamma terbaik
param_grid = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.01, 0.1, 1, 'scale', 'auto']
}

print('⏳ Melakukan Grid Search Hyperparameter Tuning (Proses ini membutuhkan waktu beberapa menit)...')
grid = GridSearchCV(
    SVC(kernel='rbf', class_weight='balanced', random_state=42),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    verbose=1
)
grid.fit(X_train, y_train)

print('\n✅ Grid Search Selesai!')
print('Best Parameters :', grid.best_params_)
print('Best F1-Score   :', grid.best_score_)

# Simpan heatmap hasil Grid Search
results = pd.DataFrame(grid.cv_results_)
scores = np.array(results.mean_test_score).reshape(4, 5)
plt.figure(figsize=(8, 6))
sns.heatmap(scores, annot=True, fmt='.4f',
            xticklabels=param_grid['gamma'], yticklabels=param_grid['C'], cmap='mako')
plt.xlabel('Gamma')
plt.ylabel('C')
plt.title('Heatmap Parameter Grid Search (USGS)', fontweight='bold')
plt.savefig('gridsearch_heatmap_usgs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Evaluasi Best Model hasil tuning secara aman
best_svm = grid.best_estimator_
y_pred_best = best_svm.predict(X_test)

print('============================================================')
print('           HASIL EVALUASI BEST MODEL (DATA RIIL USGS)')
print('============================================================')
print(f'  Akurasi (Accuracy)  : {accuracy_score(y_test, y_pred_best):.2%}')
print(f'  Presisi (Precision) : {precision_score(y_test, y_pred_best, average="weighted"):.2%}')
print(f'  Recall (Sensitivity): {recall_score(y_test, y_pred_best, average="weighted"):.2%}')
print(f'  F1-Score             : {f1_score(y_test, y_pred_best, average="weighted"):.2%}')
print('============================================================')

# Tentukan target label statis [0, 1, 2] secara aman untuk menghindari ValueError
label_names = ['Rendah', 'Sedang', 'Tinggi']
print(classification_report(y_test, y_pred_best, labels=[0, 1, 2], target_names=label_names, zero_division=0))

# Visualisasi Confusion Matrix secara aman
cm = confusion_matrix(y_test, y_pred_best, labels=[0, 1, 2])
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Absolut
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=label_names)
disp.plot(ax=axes[0], cmap='Oranges', colorbar=False)
axes[0].set_title('Confusion Matrix (Jumlah Kejadian)', fontweight='bold')

# Persentase
cm_sum = cm.sum(axis=1)[:, np.newaxis]
# Cegah pembagian dengan nol
cm_sum = np.where(cm_sum == 0, 1, cm_sum)
cm_norm = cm.astype('float') / cm_sum
sns.heatmap(cm_norm, annot=True, fmt='.2%', cmap='Oranges',
            xticklabels=label_names, yticklabels=label_names, ax=axes[1])
axes[1].set_title('Confusion Matrix (Persentase)', fontweight='bold')
axes[1].set_ylabel('Kelas Aktual')
axes[1].set_xlabel('Kelas Prediksi')

plt.tight_layout()
plt.savefig('confusion_matrix_usgs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Validasi Silang K-Fold (5-Fold & 10-Fold)
kf5 = KFold(n_splits=5, shuffle=True, random_state=42)
kf10 = KFold(n_splits=10, shuffle=True, random_state=42)

cv5_scores = cross_val_score(best_svm, X_scaled, y, cv=kf5, scoring='accuracy')
cv10_scores = cross_val_score(best_svm, X_scaled, y, cv=kf10, scoring='accuracy')

print('=======================================================')
print('       HASIL VALIDASI SILANG K-FOLD (DATA RIIL USGS)')
print('=======================================================')
print(f'  5-Fold Accuracy  : {cv5_scores.mean():.4f} +/- {cv5_scores.std():.4f}')
print(f'  10-Fold Accuracy : {cv10_scores.mean():.4f} +/- {cv10_scores.std():.4f}')
print('=======================================================')

# Plotting K-Fold
plt.figure(figsize=(10, 5))
plt.plot(range(1, 6), cv5_scores, marker='o', linestyle='-', label='5-Fold (Per Fold)')
plt.axhline(y=cv5_scores.mean(), color='orange', linestyle='--', label=f'Mean Accuracy ({cv5_scores.mean():.2%})')
plt.title('Grafik Validasi Silang 5-Fold Cross Validation (USGS)', fontweight='bold')
plt.xlabel('Lipatan (Fold)')
plt.ylabel('Akurasi')
plt.ylim([0.80, 1.01])
plt.legend()
plt.savefig('kfold_validation_usgs.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Membuat peta spasial interaktif Sesar Lembang & Jawa Barat
m = folium.Map(location=[-6.83, 107.60], zoom_start=10, control_scale=True)

# Tambahkan marker garis Sesar Lembang (koordinat perkiraan)
sesar_coords = [
    [-6.80, 107.50],
    [-6.83, 107.75]
]
folium.PolyLine(sesar_coords, color='red', weight=4, opacity=0.8, tooltip='Jalur Sesar Lembang').add_to(m)

# Tambahkan heatmap kejadian gempa
heat_data = df_lembang[['latitude', 'longitude', 'mag']].values.tolist()
HeatMap(heat_data, radius=15, blur=10, min_opacity=0.4).add_to(m)

# Simpan peta interaktif
m.save('peta_risiko_interaktif_usgs.html')
print('✅ Peta Folium interaktif berhasil disimpan sebagai peta_risiko_interaktif_usgs.html!')

In [ ]:
# Menyimpan model SVM biner dan MinMaxScaler menggunakan joblib
model_out_path = 'usgs_model.pkl'
scaler_out_path = 'usgs_scaler.pkl'

joblib.dump(best_svm, model_out_path)
joblib.dump(scaler, scaler_out_path)

# Simpan ringkasan hasil latihan ke CSV
summary_df = pd.DataFrame({
    'Parameter': ['Model', 'Kernel', 'C Terbaik', 'Gamma Terbaik', 'Total Data Latih', 'Total Data Uji', 'Akurasi', 'F1-Score', 'Recall'],
    'Nilai': ['SVM', 'RBF', str(grid.best_params_['C']), str(grid.best_params_['gamma']), str(X_train.shape[0]), str(X_test.shape[0]), f'{accuracy_score(y_test, y_pred_best):.4f}', f'{f1_score(y_test, y_pred_best, average="weighted"):.4f}', f'{recall_score(y_test, y_pred_best, average="weighted"):.4f}']
})
summary_df.to_csv('ringkasan_hasil_usgs.csv', index=False)
df_lembang.to_csv('data_lembang_usgs_dengan_prediksi.csv', index=False)

print('============================================================')
print('  EKSPOR MODEL BERHASIL!')
print('============================================================')
print(f'  Model Biner SVM  : {model_out_path}')
print(f'  Scaler Min-Max   : {scaler_out_path}')
print('============================================================')

In [ ]:
# Download berkas keluaran secara otomatis ke komputer lokal Anda
from google.colab import files

output_files = [
    'ringkasan_hasil_usgs.csv',
    'data_lembang_usgs_dengan_prediksi.csv',
    'eda_distribusi_usgs.png',
    'distribusi_kelas_usgs.png',
    'confusion_matrix_usgs.png',
    'kfold_validation_usgs.png',
    'gridsearch_heatmap_usgs.png',
    'peta_risiko_interaktif_usgs.html',
    'usgs_model.pkl',
    'usgs_scaler.pkl'
]

print('📥 Mempersiapkan pengunduhan berkas keluaran riset...')
for f in output_files:
    if os.path.exists(f):
        files.download(f)
        print(f'  ✅ Unduh berkas: {f}')
    else:
        print(f'  ⚠️ Berkas tidak ditemukan (lewati): {f}')